# MediFind PH — Source Code and Reproducibility

This notebook is the entry point for reproducing the final application using **Configuration D**. The complete FastAPI, React, ChromaDB, entity-normalization, retrieval, citation, and safety code is included beside this notebook.


## Frozen final configuration

- Knowledge base: the three official MediFind OTC PDF volumes (94 records; 226 extracted PDF pages)
- Chunk size: **800 characters**
- Chunk overlap: **200 characters**
- Retrieval depth: **top-k 3**
- Embeddings: `sentence-transformers/all-MiniLM-L6-v2`
- Vector store: ChromaDB (`medifind_otc`)
- Generator: `google/flan-t5-base`
- API: FastAPI; frontend: React/Vite


In [ ]:
from pathlib import Path
import json

ROOT = Path.cwd()
manifest = json.loads((ROOT / 'data/chroma_db/manifest.json').read_text(encoding='utf-8'))
manifest


## 1. Install backend dependencies

Run from the `03_Source_Code_and_Reproducibility` directory. Model downloads require internet access on the first run.


In [ ]:
# In a terminal (Windows PowerShell):
# cd backend
# python -m venv .venv
# .\.venv\Scripts\python.exe -m pip install -r requirements.txt


## 2. Rebuild the vector database

`build_kb.py` reads all three PDFs and uses the frozen values in `rag/config.py`. Rebuilding should yield 762 chunks when the supplied PDFs and dependency behavior are unchanged.


In [ ]:
# In a terminal from backend/:
# .\.venv\Scripts\python.exe build_kb.py


## 3. Start and verify the API


In [ ]:
# Start from backend/:
# .\.venv\Scripts\python.exe -m uvicorn main:app --reload --port 8000

# Then verify from Python:
import requests
health = requests.get('http://127.0.0.1:8000/api/health', timeout=30).json()
health


In [ ]:
payload = {'message': 'What is the generic name of Dolfenal?'}
response = requests.post('http://127.0.0.1:8000/api/chat', json=payload, timeout=120).json()
response


## 4. Run backend regression suites

These cover catalog integrity, formulation narrowing, comparisons, warning/leaflet sections, citations, retrieval, safety gates, and the Configuration D invariant.


In [ ]:
# In a terminal from backend/:
# .\.venv\Scripts\python.exe test_catalog.py
# .\.venv\Scripts\python.exe test_formulations.py
# .\.venv\Scripts\python.exe test_gaviscon.py
# .\.venv\Scripts\python.exe test_comparisons.py
# .\.venv\Scripts\python.exe test_leaflets.py
# .\.venv\Scripts\python.exe test_retrieval.py
# .\.venv\Scripts\python.exe test_regression.py


## 5. Start the React frontend


In [ ]:
# In a second terminal from frontend/:
# npm ci
# npm run dev


## 6. Re-run the fixed chatbot evaluation suite

From the repository root, run `node experiments/Chatbot_Accuracy_Evaluation_Deliverable/run_evaluation.mjs` only while the intended API version is running. The included submission results are the preserved baseline live run and must not be relabeled as post-fix evidence. Final Phase 5 manual testing remains the responsibility of the researchers.
